# 🔨 Development el dood — APK Builder
شغّل كل cell بالترتيب واحدة ورا واحدة.
**وقت البناء الكلي: ~20-40 دقيقة**

In [ ]:
#@title ✅ Cell 1 — تحديث النظام وتثبيت System Dependencies
import subprocess, sys

print('📦 تثبيت system packages...')
subprocess.run([
    'apt-get', 'install', '-y',
    'git', 'zip', 'unzip', 'wget', 'curl',
    'openjdk-17-jdk',
    'autoconf', 'automake', 'libtool', 'pkg-config',
    'zlib1g-dev', 'libncurses5-dev', 'libncursesw5-dev',
    'libffi-dev', 'libssl-dev', 'libsqlite3-dev',
    'libbz2-dev', 'libreadline-dev', 'libgdbm-dev',
    'liblzma-dev', 'python3-dev',
    'build-essential', 'ccache', 'libltdl-dev',
    'cmake', 'ninja-build', 'patch',
    'lib32stdc++6', 'lib32z1'
], check=True)
print('✅ System packages جاهزة')

In [ ]:
#@title ✅ Cell 2 — تثبيت Buildozer و Cython
import subprocess

print('🐍 تثبيت Buildozer + Cython...')
subprocess.run(['pip', 'install', '--upgrade', 'pip', 'wheel'], check=True)
subprocess.run(['pip', 'install', 'buildozer==1.5.0', 'cython==0.29.37'], check=True)
print('✅ Buildozer و Cython جاهزين')

In [ ]:
#@title ✅ Cell 3 — تحميل Android Command Line Tools
import subprocess, os

ANDROID_HOME = '/root/android-sdk'
CMDLINE_TOOLS_URL = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'

os.makedirs(f'{ANDROID_HOME}/cmdline-tools', exist_ok=True)

print('⬇️ تحميل Android command line tools...')
subprocess.run(['wget', '-q', CMDLINE_TOOLS_URL, '-O', '/tmp/cmdtools.zip'], check=True)
subprocess.run(['unzip', '-q', '/tmp/cmdtools.zip', '-d', f'{ANDROID_HOME}/cmdline-tools'], check=True)

# إعادة تسمية المجلد إلى latest
old_path = f'{ANDROID_HOME}/cmdline-tools/cmdline-tools'
new_path = f'{ANDROID_HOME}/cmdline-tools/latest'
if os.path.exists(old_path):
    os.rename(old_path, new_path)

# تصدير متغيرات البيئة
os.environ['ANDROID_HOME'] = ANDROID_HOME
os.environ['ANDROID_SDK_ROOT'] = ANDROID_HOME
SDKMANAGER = f'{ANDROID_HOME}/cmdline-tools/latest/bin/sdkmanager'

print(f'✅ Android tools جاهزة في: {ANDROID_HOME}')

In [ ]:
#@title ✅ Cell 4 — قبول Licenses وتثبيت NDK 25b
import subprocess, os

ANDROID_HOME = '/root/android-sdk'
SDKMANAGER = f'{ANDROID_HOME}/cmdline-tools/latest/bin/sdkmanager'
os.environ['ANDROID_HOME'] = ANDROID_HOME
os.environ['ANDROID_SDK_ROOT'] = ANDROID_HOME

print('📜 قبول SDK licenses...')
subprocess.run(
    f'yes | {SDKMANAGER} --licenses',
    shell=True, capture_output=True
)

print('⬇️ تثبيت NDK 25b و build tools (هياخد وقت)...')
subprocess.run([
    SDKMANAGER,
    'ndk;25.1.8937393',
    'build-tools;33.0.2',
    'platforms;android-33',
    'platform-tools'
], check=True)

NDK_PATH = f'{ANDROID_HOME}/ndk/25.1.8937393'
os.environ['ANDROIDNDK'] = NDK_PATH
os.environ['ANDROIDSDK'] = ANDROID_HOME
print(f'✅ NDK جاهز في: {NDK_PATH}')

In [ ]:
#@title ✅ Cell 5 — جلب الكود من GitHub
import subprocess, os

print('📥 جلب المشروع من GitHub...')
subprocess.run([
    'git', 'config', '--global',
    'url.https://.insteadOf', 'git://'
], check=True)

if os.path.exists('/root/Password-Generator'):
    subprocess.run(['git', '-C', '/root/Password-Generator', 'pull'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/mahmoueda769-crypto/Password-Generator.git',
        '/root/Password-Generator'
    ], check=True)

os.chdir('/root/Password-Generator')
print(f'✅ الكود جاهز في: {os.getcwd()}')
subprocess.run(['ls', '-la'], check=True)

In [ ]:
#@title ✅ Cell 6 — حقن مسارات SDK/NDK في buildozer.spec
import os

ANDROID_HOME = '/root/android-sdk'
NDK_PATH = f'{ANDROID_HOME}/ndk/25.1.8937393'

os.chdir('/root/Password-Generator')

with open('buildozer.spec', 'r') as f:
    content = f.read()

# إزالة أي sdk_path/ndk_path قديمة
lines = [l for l in content.splitlines()
         if not l.startswith('android.sdk_path') and
            not l.startswith('android.ndk_path')]

# حقن المسارات قبل [buildozer]
new_lines = []
for line in lines:
    if line.strip() == '[buildozer]':
        new_lines.append(f'android.sdk_path = {ANDROID_HOME}')
        new_lines.append(f'android.ndk_path = {NDK_PATH}')
        new_lines.append('')
    new_lines.append(line)

with open('buildozer.spec', 'w') as f:
    f.write('\n'.join(new_lines))

print('✅ buildozer.spec بعد الحقن:')
print(open('buildozer.spec').read())

In [ ]:
#@title 🔨 Cell 7 — بناء الـ APK (الخطوة الأهم — هتاخد وقت)
import subprocess, os

ANDROID_HOME = '/root/android-sdk'
os.environ['ANDROID_HOME'] = ANDROID_HOME
os.environ['ANDROID_SDK_ROOT'] = ANDROID_HOME
os.environ['ANDROIDSDK'] = ANDROID_HOME
os.environ['ANDROIDNDK'] = f'{ANDROID_HOME}/ndk/25.1.8937393'
os.environ['ANDROIDAPI'] = '33'
os.environ['ANDROIDMINAPI'] = '21'
os.environ['PATH'] += f':{ANDROID_HOME}/platform-tools'

os.chdir('/root/Password-Generator')

print('🚀 بناء الـ APK... (20-40 دقيقة)')
result = subprocess.run(
    ['buildozer', '-v', 'android', 'debug'],
    capture_output=False
)

if result.returncode == 0:
    print('\n✅✅✅ تم بناء الـ APK بنجاح!')
else:
    print('\n❌ فشل البناء — اقرأ الـ log فوق')

In [ ]:
#@title ⬇️ Cell 8 — تحميل الـ APK على جهازك
import os, glob
from google.colab import files

os.chdir('/root/Password-Generator')
apks = glob.glob('bin/*.apk')

if apks:
    print(f'✅ تم إيجاد الـ APK: {apks[0]}')
    print('⬇️ جاري التحميل...')
    files.download(apks[0])
else:
    print('❌ مفيش APK — ارجع لـ Cell 7 وشوف الـ error')